# CoT Enrichment Audit

Stage D probe of teacher-enriched traces from DeepSeek-V4-Flash. Run on the 10-id smoke test (job 1058498), then re-point at the production output once it exists.

Checks:
1. Parse failures — raw output of every `parse_ok=False` row, diagnose JSON-extraction issues
2. Forward-reasoning leak — grep enriched CoTs for answer-revealing phrases
3. Score distributions per dimension
4. Strategy distribution and gate decisions
5. Spot-check kept traces (full inspection of N=3)

In [ ]:
import json, re, random
from pathlib import Path
from collections import Counter

OUT = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/teacher_enriched_1058498_TEST.jsonl')
STAGE_A = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/cot_1058163_Qwen3-VL-235B-A22B-Thinking-FP8_train_N16_T0.8_grounding.jsonl')

rows = [json.loads(l) for l in OUT.open()]
print(f'{len(rows)} rows; parse_ok={sum(r["parse_ok"] for r in rows)}/{len(rows)}')

## 1. Parse failures

In [ ]:
fails = [r for r in rows if not r['parse_ok']]
print(f'{len(fails)} parse failures')
for r in fails:
    print('=' * 80)
    print(f'id={r["id"]}')
    print('--- raw tail (last 1500 chars) ---')
    print(r['raw'][-1500:])
    print()

## 2. Forward-reasoning leak check

Enriched CoT must read as forward reasoning, not reverse-engineering from the gold answer.

In [ ]:
LEAK_PATTERNS = [
    r'since the answer is',
    r'given that the answer',
    r'we know the answer',
    r'because the answer is',
    r'the answer (is|must be) given',
    r'gold answer',
    r'working backward',
    r'work(ing)? back(ward)?',
]
leak_re = re.compile('|'.join(f'({p})' for p in LEAK_PATTERNS), re.IGNORECASE)

leaks = []
for r in rows:
    if not r['parse_ok']:
        continue
    cot = r['parsed'].get('enriched_cot', '') or ''
    m = leak_re.search(cot)
    if m:
        leaks.append((r['id'], m.group(0), cot))

print(f'{len(leaks)} traces with leak patterns')
for id_, hit, cot in leaks:
    print('=' * 80)
    print(f'id={id_} hit="{hit}"')
    idx = cot.lower().find(hit.lower())
    print(cot[max(0, idx-200): idx+200])

## 3. Score distributions

In [ ]:
dims = ['faithfulness', 'logical_validity', 'completeness', 'conciseness']
from collections import defaultdict
by_dim = defaultdict(list)
for r in rows:
    if not r['parse_ok']:
        continue
    scores = r['parsed'].get('scores', {})
    for d in dims:
        s = scores.get(d, {}).get('score')
        if isinstance(s, (int, float)):
            by_dim[d].append(s)

for d in dims:
    vals = by_dim[d]
    if vals:
        print(f'{d:18s}  n={len(vals)}  mean={sum(vals)/len(vals):.2f}  hist={Counter(vals)}')
    else:
        print(f'{d:18s}  no scores')

## 4. Strategy & gate

In [ ]:
strats = Counter()
keeps = Counter()
match_gold = Counter()
for r in rows:
    if not r['parse_ok']:
        strats['_parse_fail'] += 1
        continue
    p = r['parsed']
    strats[p.get('enrichment_strategy', '?')] += 1
    keeps[bool(p.get('keep_for_sft'))] += 1
    match_gold[bool(p.get('self_check_matches_gold'))] += 1

print('strategies     :', dict(strats))
print('keep_for_sft   :', dict(keeps))
print('matches_gold   :', dict(match_gold))

## 5. Spot-check 3 random kept traces (full content)

In [ ]:
kept = [r for r in rows if r['parse_ok'] and r['parsed'].get('keep_for_sft')]
print(f'{len(kept)} kept; sampling 3 (or fewer)')
rng = random.Random(0)
for r in rng.sample(kept, min(3, len(kept))):
    p = r['parsed']
    print('=' * 80)
    print(f'id={r["id"]}  strategy={p.get("enrichment_strategy")}  selected_idx={p.get("selected_candidate_idx")}')
    print(f'final_answer={p.get("final_answer")!r}')
    print(f'self_check_matches_gold={p.get("self_check_matches_gold")}')
    print('--- enriched_cot ---')
    print(p.get('enriched_cot', ''))
    print('--- scores ---')
    for d in dims:
        s = p.get('scores', {}).get(d, {})
        print(f'  {d}: {s.get("score")} — {s.get("justification")}')

## 6. Cross-check against gold (Stage A)

Pull each id's gold from Stage A and compare to teacher's `final_answer`.

In [ ]:
ids = {r['id'] for r in rows}
gold = {}
with STAGE_A.open() as f:
    for line in f:
        if not line.endswith('\n'):
            break
        s = json.loads(line)
        if s['id'] in ids and s['id'] not in gold:
            gold[s['id']] = s['gt_answer']
        if len(gold) == len(ids):
            break

agree = disagree = 0
for r in rows:
    if not r['parse_ok']:
        continue
    g = str(gold.get(r['id'], '')).strip()
    a = str(r['parsed'].get('final_answer', '')).strip()
    if g and a and g.lower() == a.lower():
        agree += 1
    else:
        disagree += 1
        print(f'mismatch id={r["id"]}  gold={g!r}  teacher={a!r}')
print(f'\nagree={agree}  disagree={disagree}')